In [1]:
import torch, transformers
import json

model_8B_id = "meta-llama/Meta-Llama-3-8B-Instruct"
model_3B_id = "meta-llama/Llama-3.2-3B-Instruct"
tok = transformers.AutoTokenizer.from_pretrained(model_8B_id)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_8B_id, torch_dtype=torch.float16, device_map="cuda"
)

d:\GeoTKG\llama_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:23<00:00,  5.90s/it]


In [2]:
def get_test_data():
    examples = []
    with open("D:\\GeoTKG\\cleandata\\tie\\test.json", "r") as f:
        examples=[json.loads(line) for line in f]
    return examples

def get_ner_prompt(text, method):
    if method == "event and time":
        tags = "EVENT, DATE, TIME, DURATION, SET"
    elif method == "geoscience":
        tags = "LOCATION, MINERAL, ORE_DEPOSIT, ROCK, STRAT, TIMESCALE"
    sys = f'''You are a Named Entity Recognition (NER) system for tagging {method} entities. Identify and classify entities in the text based on the entity types: {tags}. 
            Each entity should be represented as a tuple (entity surface text, type) in valid JSON format.
            Return ONLY valid JSON (no markdown, no commentary). Escape double quotes as \".'''
    
    user = f'''Extract entities from the following text: {text}'''
    messages = [
        {"role": "system", "content": sys},
        {"role": "user", "content": user}
    ]
    return messages

def get_norm_prompt(text, dct):
    sys = '''
        You are a geological timescale and calendar time normalization system. Normalize the time expressions that have been tagged in the text using the document creation time as anchor for calendar time.
        Each time mention should be represented as a tuple: (surface text, normalized value).
        Geological timescale normalized values should be in ma (million years ago).
        Calendar time expressions should be in ISO 8601 format (YYYY-MM-DD).
        Return ONLY valid JSON (no markdown, no commentary). Escape double quotes as \".
    '''
    user = f'Normalize time expressions from the following passage which has a document creation time of {dct}: {text}'
    messages = [
    {"role":"system","content":sys},
    {"role":"user","content":user}
    ]
    return messages

def get_tkg_prompt(text, dct):
    text = " ".join([wrd for sent in text for wrd in sent])
    sys = '''
        You are an information extraction system for geoscience and general texts.
        Extract (1) times, (2) quintuples (events), and (3) temporal-relation triples.
        Return ONLY valid JSON (no markdown, no commentary). Escape double quotes as \".

        Gregorian Calendar Times and Geological Timescales
        - Each time has format: ["T#", "surface text", "normalized value or null", "DATE|DURATION|SET|TIME|GEO_TIME"]
        - Reuse the same T# if the surface text repeats.

        Quintuples (events)
        - Each event is one quintuple: ["E#", "subject or null", "event string", "object or null", "T# or null", "T# or null"]
        - Event string must be short (just the trigger words).
        - E# assigned in order of first mention; reuse IDs for duplicates.

        Temporal triples
        - BEFORE: event1 ends before event2 starts
        - AFTER:  event1 starts after event2 ends
        - DURING: event1 occurs fully within event2
        - CONTAINS: event1 fully contains event2
        - IDENTITY/EQUALS: same event/time span
        - OVERLAPS: partial intersection
        - Each relation is ["E#", "BEFORE|AFTER|DURING|CONTAINS|IDENTITY|EQUALS|OVERLAPS", "E#"]
        - Only E# allowed, never T#.
        - Each unordered event pair appears at most once.

        Validation
        - IDs sequential by first mention (E1, E2 ...; T1, T2 ...).
        - All T# in quintuples must exist in times.
        - JSON must be valid: no trailing commas, no comments.
    '''
    user = f'Extract events and temporal relations from the following passage (document creation time: {dct}):    {text}'
    messages = [
    {"role":"system","content":sys},
    {"role":"user","content":user}
    ]
    return messages

def jsonify(output):
    try:
        return json.loads(output)
    except json.JSONDecodeError:
        try:
            return json.loads(output+"}")
        except json.JSONDecodeError:
            return None

def inference(model, tok, prompt, max_new_tokens=4000):
    input_prompt = tok.apply_chat_template(prompt, add_generation_prompt=True, tokenize=False)
    inputs = tok(input_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.2, do_sample=False)
    gen_tokens = out[0, inputs.input_ids.shape[-1]:]
    decodings = tok.decode(gen_tokens, skip_special_tokens=True)
    prediction = decodings.strip(":\n`").lstrip("```json\n").rstrip("\n```")
    return prediction

def norm_preprocess(text, times):
    out = ""
    for sn, sent in enumerate(text):
        sent_times = [t['offset'] for t in times if t['sent_id'] == sn]
        sent_times = sorted(sent_times, key=lambda x: x[0], reverse=True)

        for st, en in sent_times:
            sent.insert(en,"</timex>")
            sent.insert(st,"<timex>")
        out += " ".join(sent) + " "
    return out.strip()

def post_processing(preds):
    out = {}
    for fn, pred in enumerate(preds):
        json_out = jsonify(pred['pred'])
        if json_out is None:
            testy = pred['pred'].partition('{')[-1].rpartition('}')[0]
            json_out = jsonify('{'+testy+'}')
        
        if json_out is None or json_out == {}:
            testy = pred['pred'].partition('{')[-1][:-6]
            json_out = jsonify('{'+testy+'}')

        if json_out is None:
            testy = pred['pred'].partition('{')[-1][:-1]
            json_out = jsonify('{'+testy+'}')

        if json_out is None:
            testy = pred['pred'].partition('{')[-1].rpartition(']')[0]
            json_out = jsonify('{'+testy+'}')

        if type(json_out) is list:
            json_out = json_out[0]

        keys = list(json_out.keys())
        keys.remove('times')
        keys.remove('quintuples')
        json_out['triples'] = json_out.pop(keys[0])

        out[fn] = {'text':pred['text'], 'pred':json_out}

def ner_post_processing(ner_preds):
    poster = []

    for fn, pred in enumerate(ner_preds):
        if fn == 121:
            poster.append(pred)
        else:
            texty = pred['pred'].partition('[')[-1].rpartition(']')[0]
            tes = jsonify("["+texty+"]")

            if tes is None:
                texty = pred['pred'].partition('[')[-1].rpartition(']')[0].replace("{", "[").replace("}", "]")
                tes = jsonify("["+texty+"]")
            
            if tes is None:
                tes = texty

            poster.append({'text':pred['text'], 'pred':tes})
    return poster


In [3]:
test_data = get_test_data()

In [4]:
prediction_type = "norm"

if prediction_type == "tkg":
    chat_tkg_preds = []
    file_num = 1
    for example in test_data:
        dct = [inst['value'] for inst in example['instances'] if inst['type'] != "EVENT" and inst['id'] == 0][0]
        prompt = get_tkg_prompt(example['text'], dct)
        prediction = inference(model, tok, prompt)
        chat_tkg_preds.append({"text":example['text'], "pred":prediction})
        print(f"Processed example {file_num} / {len(test_data)}")
        file_num += 1
elif prediction_type == "ner":
    print("----- RUNNING NER PREDICTIONS -----")
    chat_ner_preds = []
    out_file_name = "llama3.2-8B-ner-preds.json"
    file_num = 1
    for example in test_data[121:122]:
        prompt = get_ner_prompt(example['text'], "event and time")
        prediction = inference(model, tok, prompt, max_new_tokens=1000)
        chat_ner_preds.append({"text":example['text'], "pred":prediction})
        print(f"Processed example {file_num} / {len(test_data)}")
        file_num += 1
elif prediction_type == "norm":
    chat_norm_preds = []
    out_file_name = "llama3.2-8B-norm-preds.json"
    file_num = 1
    for example in test_data:
        dct = [inst['value'] for inst in example['instances'] if inst['type'] != "EVENT" and inst['id'] == 0][0]
        prompt = get_norm_prompt(norm_preprocess(example['text'], [instance for instance in example['instances'] if instance['type'] != "EVENT" and instance['id'] != 0]), dct)
        prediction = inference(model, tok, prompt, max_new_tokens=1000)
        chat_norm_preds.append({"text":example['text'], "pred":prediction})
        print(f"Processed example {file_num} / {len(test_data)}")
        file_num += 1

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed example 1 / 151


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed example 2 / 151


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Processed example 148 / 151


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed example 149 / 151


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed example 150 / 151
Processed example 151 / 151


In [ ]:
failed_times = []
failed_events = []
failed_trips = []
for fn, pred in enumerate(out):
    if out[pred]['pred'] is None:
        failed_times.append(fn)
        failed_events.append(fn)
        failed_trips.append(fn)
    else:
        try:
            x = out[pred]['pred']['times']
        except KeyError:
            failed_times.append(fn)

        try:
            y = out[pred]['pred']['quintuples']
        except KeyError:
            failed_events.append(fn)

        try:
            z = out[pred]['pred']['triples']
        except KeyError:
            failed_trips.append(fn)

failed_times, failed_events, failed_trips

In [19]:
import json
with open("llama3.2-8B-norm-preds.json", 'w') as json_file:
    for sample in chat_norm_preds:
        json_file.write(json.dumps(sample)+"\n")

In [18]:
for fn, uh in enumerate(chat_norm_preds):
    tes_json = jsonify(uh['pred'])
    if tes_json == None:
        print(fn, uh['pred'])


5 [
  ("Thursday", "1998-08-06T00:00:00"),
  ("Aug. 7", "1998-08-07T00:00:00")
]
7 [
  ("Friday", "1998-08-14T00:00:00"),
  ("this month", "1998-08-01T00:00:00"),
  ("Thursday", "1998-08-06T00:00:00"),
  ("Aug. 7", "1998-08-07T00:00:00")
]
10 [
  ("Wednesday", "1998-09-30"),
  ("Aug. 7", "1998-08-07"),
  ("1995", "1995-04-19"),
  ("Thursday", "1998-09-31"),
  ("Friday", "1998-10-02")
]
11 [
  ("Saturday", "1998-12-05T00:00:00"),
  ("early this week", "1998-12-01T00:00:00"),
  ("April", "1999-04-01T00:00:00"),
  ("April", "1999-04-01T00:00:00")
]
17 [
  ("last fall", "1998-10-01"),
  ("Today", "1999-05-07T00:00:00"),
  ("November", "1998-11-01"),
  ("Thursday", "1999-05-06"),
  ("Nov. 3", "1998-11-03"),
  ("last month", "1999-04-01"),
  ("Thursday", "1999-05-06"),
  ("1990", "0.0"),
  ("weeks", "1999-01-01"),
  ("December", "1998-12-01"),
  ("1994", "4.5"),
  ("1997", "6.5"),
  ("January 1998", "1998-01-01")
]
19 [
  ("last year", "1998-10-08"),
  ("midday", "1999-10-08T12:00:00"),
  ("

In [ ]:
chat_norm_preds